# Custom User Model & Signals

## AbstractUser vs AbstractBaseUser

| Feature | `AbstractBaseUser` | `AbstractUser` |
|--------|-------------------|----------------|
| Built-in fields | None | Includes `email`, `first_name`, etc. |
| Login system | Must implement everything | Works out of the box |
| Admin compatibility | Must customize | Mostly automatic |
| Use case | Full rebuild (e.g. RFID or mobile login) | Extend the existing system |

When you only need to change the login field (e.g. use mobile number), `AbstractUser` is the right choice.


## Custom User Model

```python
# accounts/models.py
from django.contrib.auth.models import AbstractUser
from django.db import models

class CustomUser(AbstractUser):
    mobile = models.CharField(max_length=15, unique=True)

    USERNAME_FIELD = 'mobile'   # login with mobile
    REQUIRED_FIELDS = []

    def save(self, *args, **kwargs):
        if not self.username:
            self.username = self.mobile  # keep username in sync
        super().save(*args, **kwargs)
```

Register in `settings.py`:
```python
AUTH_USER_MODEL = 'accounts.CustomUser'
```

Always set `AUTH_USER_MODEL` before the first migration. Changing it later is complex.


## Custom User Manager

```python
# accounts/models.py
from django.contrib.auth.models import BaseUserManager

class CustomUserManager(BaseUserManager):
    def create_user(self, mobile, password=None, **extra_fields):
        if not mobile:
            raise ValueError("Mobile number is required")
        user = self.model(mobile=mobile, **extra_fields)
        user.set_password(password)
        user.save()
        return user

    def create_superuser(self, mobile, password=None, **extra_fields):
        extra_fields.setdefault('is_staff', True)
        extra_fields.setdefault('is_superuser', True)
        return self.create_user(mobile, password, **extra_fields)
```

Attach the manager inside `CustomUser`:
```python
objects = CustomUserManager()
```


## Django Signals

Signals implement the Observer Pattern: when an event occurs (e.g. a model is saved), registered listeners are called automatically.

### Sending a Welcome Email on Signup

```python
# accounts/signals.py
from django.db.models.signals import post_save
from django.dispatch import receiver
from django.core.mail import send_mail
from .models import CustomUser

@receiver(post_save, sender=CustomUser)
def send_welcome_email(sender, instance, created, **kwargs):
    if created:
        send_mail(
            subject='Welcome!',
            message='Thanks for signing up.',
            from_email='no-reply@example.com',
            recipient_list=[f'{instance.mobile}@example.com'],
        )
```

### Connecting Signals in apps.py

```python
# accounts/apps.py
from django.apps import AppConfig

class AccountsConfig(AppConfig):
    name = 'accounts'

    def ready(self):
        import accounts.signals
```

Use the console email backend during development:
```python
EMAIL_BACKEND = 'django.core.mail.backends.console.EmailBackend'
```


## Auto-Creating a Profile with Signals

```python
# accounts/models.py
class Profile(models.Model):
    user = models.OneToOneField(CustomUser, on_delete=models.CASCADE)
    bio = models.TextField(blank=True)
    avatar = models.ImageField(upload_to='avatars/', blank=True)
```

```python
# accounts/signals.py (add below the email signal)
from .models import Profile

@receiver(post_save, sender=CustomUser)
def create_profile(sender, instance, created, **kwargs):
    if created:
        Profile.objects.create(user=instance)
```

**Signal parameters:**
- `sender` — the model class
- `instance` — the saved object
- `created` — `True` only for new objects


## Signup View

```python
# accounts/views.py
from django.shortcuts import render, redirect
from django.contrib.auth import login
from .models import CustomUser

def signup_view(request):
    if request.method == 'POST':
        mobile = request.POST['mobile']
        password = request.POST['password']
        user = CustomUser.objects.create_user(mobile=mobile, password=password)
        login(request, user)
        return redirect('home')
    return render(request, 'signup.html')
```

```html
<!-- signup.html -->
<form method="post">
  {% csrf_token %}
  <input type="text" name="mobile" placeholder="Mobile"><br>
  <input type="password" name="password" placeholder="Password"><br>
  <button type="submit">Register</button>
</form>
```


## Summary

- Use `AbstractUser` when extending the default user system; use `AbstractBaseUser` only for a complete rebuild.
- Set `USERNAME_FIELD` to change the login field and define `AUTH_USER_MODEL` before the first migration.
- `BaseUserManager` provides `create_user()` and `create_superuser()` methods.
- Signals decouple side effects (email, profile creation) from the main save logic.
- `post_save` with `created=True` fires only when a new object is created, not on updates.
- Connect signals in `AppConfig.ready()` to ensure they load at startup.
